# Round 1 Data Exploration

Products: **ASH_COATED_OSMIUM**, **INTARIAN_PEPPER_ROOT**  
Days: -2, -1, 0

In [40]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), ""))
from plotter import Plotter

DATA_DIR = "../data/ROUND1"
days = [-2, -1, 0]

In [41]:
import pandas as pd

In [42]:
all_prices = pd.concat(
    [pd.read_csv(f"{DATA_DIR}/prices_round_1_day_{d}.csv", delimiter=";") for d in days],
    ignore_index=True,
)

osmium = all_prices[all_prices["product"] == "ASH_COATED_OSMIUM"].reset_index(drop=True)
pepper = all_prices[all_prices["product"] == "INTARIAN_PEPPER_ROOT"].reset_index(drop=True)

print(f"osmium: {len(osmium)} rows, pepper: {len(pepper)} rows")

osmium: 30000 rows, pepper: 30000 rows


In [43]:
pepper['spread'] = pepper['ask_price_1'] - pepper['bid_price_1']
osmium['spread'] = osmium['ask_price_1'] - osmium['bid_price_1']

In [44]:
# Load trades with day column (trades CSVs don't have one, so we add it)
all_trades = pd.concat(
    [pd.read_csv(f"{DATA_DIR}/trades_round_1_day_{d}.csv", delimiter=";")
       .assign(day=d)
     for d in days],
    ignore_index=True,
)

# Merge all price columns onto trades at matching (day, timestamp, product)
all_trades = all_trades.rename(columns={"symbol": "product"})
all_trades = all_trades.merge(all_prices, on=["day", "timestamp", "product"], how="left")

osmium_trades = all_trades[all_trades["product"] == "ASH_COATED_OSMIUM"].reset_index(drop=True)
pepper_trades = all_trades[all_trades["product"] == "INTARIAN_PEPPER_ROOT"].reset_index(drop=True)

print(f"osmium_trades: {len(osmium_trades)} rows, {list(osmium_trades.columns)}")
print(f"pepper_trades: {len(pepper_trades)} rows")
pepper_trades.head()

osmium_trades: 1265 rows, ['timestamp', 'buyer', 'seller', 'product', 'currency', 'price', 'quantity', 'day', 'bid_price_1', 'bid_volume_1', 'bid_price_2', 'bid_volume_2', 'bid_price_3', 'bid_volume_3', 'ask_price_1', 'ask_volume_1', 'ask_price_2', 'ask_volume_2', 'ask_price_3', 'ask_volume_3', 'mid_price', 'profit_and_loss']
pepper_trades: 1011 rows


,timestamp,buyer,seller,product,currency,price,quantity,day,bid_price_1,bid_volume_1,...,bid_price_3,bid_volume_3,ask_price_1,ask_volume_1,ask_price_2,ask_volume_2,ask_price_3,ask_volume_3,mid_price,profit_and_loss
0,1000,NaN,NaN,INTARIAN_PEPPER_ROOT,XIRECS,9995.0,7,-2,9995.0,11.0,...,NaN,NaN,10006.0,11.0,10009.0,24.0,NaN,NaN,10000.5,0.0
1,4000,NaN,NaN,INTARIAN_PEPPER_ROOT,XIRECS,10007.0,7,-2,10007.0,7.0,...,9996.0,20.0,10010.0,11.0,10012.0,20.0,NaN,NaN,10008.5,0.0
2,10500,NaN,NaN,INTARIAN_PEPPER_ROOT,XIRECS,10013.0,5,-2,10013.0,5.0,...,10002.0,22.0,10016.0,9.0,10019.0,22.0,NaN,NaN,10014.5,0.0
3,12600,NaN,NaN,INTARIAN_PEPPER_ROOT,XIRECS,10018.0,3,-2,10007.0,12.0,...,NaN,NaN,10018.0,12.0,10021.0,16.0,NaN,NaN,10012.5,0.0
4,15000,NaN,NaN,INTARIAN_PEPPER_ROOT,XIRECS,10021.0,7,-2,10009.0,12.0,...,NaN,NaN,10021.0,12.0,10023.0,25.0,NaN,NaN,10015.0,0.0


In [45]:
pepper['spread'].value_counts()

spread
13.0    6355
12.0    6138
14.0    4752
11.0    3069
16.0    2058
15.0    1895
17.0    1574
3.0      373
2.0      282
18.0     219
19.0     171
8.0      141
4.0      138
20.0     133
9.0      128
10.0      67
7.0       66
6.0       57
5.0       49
21.0      23
Name: count, dtype: int64

In [46]:
pepper_trades['spread'] = pepper_trades['ask_price_1'] - pepper_trades['bid_price_1']

In [47]:
pepper_trades['spread'].value_counts()

spread
12.0    160
13.0    144
3.0     132
14.0    123
2.0     100
11.0     87
16.0     50
15.0     47
4.0      36
17.0     34
5.0      13
6.0      13
9.0       8
8.0       6
19.0      4
7.0       3
18.0      3
10.0      3
20.0      2
21.0      1
Name: count, dtype: int64

In [48]:
pepper_take_on_buy = pepper_trades[
    (pepper_trades['bid_price_1'] >= pepper_trades['price']) &
    (pepper_trades['ask_price_1'] - pepper_trades['price'] >= 12)
]
pepper_take_on_sell = pepper_trades[
    (pepper_trades['ask_price_1'] <= pepper_trades['price']) &
    (pepper_trades['price'] - pepper_trades['bid_price_1'] >= 12)
]
pepper_neither = pepper_trades[
    (pepper_trades['ask_price_1'] > pepper_trades['price']) &
    (pepper_trades['bid_price_1'] < pepper_trades['price'])
]
print(pepper_take_on_buy.shape, pepper_take_on_sell.shape, pepper_neither.shape)
# there is 313 possible takes on buys (meaning we would be selling)
# there are 255 takes on sells (meaning we would be buying)
# 

(313, 23) (255, 23) (1, 23)


In [49]:
pepper_take_on_buy['quantity'].sum()

np.int64(1590)

In [50]:
pepper_take_on_sell['quantity'].sum()

np.int64(1292)

In [ ]:
import plotly.graph_objects as go

DAY_OFFSET = 1_000_000

buy_sorted = pepper_take_on_buy.sort_values(['day', 'timestamp']).copy()
sell_sorted = pepper_take_on_sell.sort_values(['day', 'timestamp']).copy()

buy_sorted['global_ts'] = buy_sorted['timestamp'] + (buy_sorted['day'] - buy_sorted['day'].min()) * DAY_OFFSET
sell_sorted['global_ts'] = sell_sorted['timestamp'] + (sell_sorted['day'] - sell_sorted['day'].min()) * DAY_OFFSET

buy_cum = buy_sorted.groupby('global_ts')['quantity'].sum().cumsum()
sell_cum = sell_sorted.groupby('global_ts')['quantity'].sum().cumsum()

# individual trade quantities at each timestamp
buy_per_ts = buy_sorted.groupby('global_ts')['quantity'].sum()
sell_per_ts = sell_sorted.groupby('global_ts')['quantity'].sum()

fig = go.Figure()
fig.add_trace(go.Scattergl(x=buy_cum.index, y=buy_cum.values, mode='lines', name='cumulative take on buy (sells)', line=dict(color='red')))
fig.add_trace(go.Scattergl(x=sell_cum.index, y=sell_cum.values, mode='lines', name='cumulative take on sell (buys)', line=dict(color='green')))
fig.add_trace(go.Scattergl(x=buy_cum.index, y=buy_cum.values, mode='markers', name='buy trades', marker=dict(color='red', size=5+10*(buy_per_ts/buy_per_ts.max()), opacity=0.6), customdata=buy_per_ts.values, hovertemplate='ts: %{x}<br>cumulative: %{y}<br>qty: %{customdata}<extra></extra>'))
fig.add_trace(go.Scattergl(x=sell_cum.index, y=sell_cum.values, mode='markers', name='sell trades', marker=dict(color='green', size=5+10*(sell_per_ts/sell_per_ts.max()), opacity=0.6), customdata=sell_per_ts.values, hovertemplate='ts: %{x}<br>cumulative: %{y}<br>qty: %{customdata}<extra></extra>'))
fig.update_layout(title='Pepper: cumulative take quantity over time', xaxis_title='global timestamp', yaxis_title='cumulative quantity')
fig.show(renderer='browser')

## Day -2

In [22]:
df = pd.read_csv(f"{DATA_DIR}/prices_round_1_day_-2.csv")

In [24]:
p = Plotter(f"{DATA_DIR}/prices_round_1_day_-2.csv", f"{DATA_DIR}/trades_round_1_day_-2.csv")
p.visualize_orderbook(product="INTARIAN_PEPPER_ROOT", ymin=9990, ymax=10500, renderer="browser")

Task was destroyed but it is pending!
task: <Task pending name='Task-215' coro=<_async_in_context.<locals>.run_in_context() done, defined at /home/utkarsh/Documents/codebase/IMCProsperity_2026/.venv/lib/python3.13/site-packages/ipykernel/utils.py:57> wait_for=<Task pending name='Task-216' coro=<Kernel.shell_main() running at /home/utkarsh/Documents/codebase/IMCProsperity_2026/.venv/lib/python3.13/site-packages/ipykernel/kernelbase.py:597> cb=[Task.task_wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at /home/utkarsh/Documents/codebase/IMCProsperity_2026/.venv/lib/python3.13/site-packages/zmq/eventloop/zmqstream.py:563]>
/home/utkarsh/Documents/codebase/IMCProsperity_2026/.venv/lib/python3.13/site-packages/_plotly_utils/utils.py:431: RuntimeWarning: coroutine 'Kernel.shell_main' was never awaited
  map(
Task was destroyed but it is pending!
task: <Task pending name='Task-216' coro=<Kernel.shell_main() running at /home/utkarsh/Documents/codebase/IMCProsperity_2026/.venv/lib/

## Day -1

In [8]:
p = Plotter(f"{DATA_DIR}/prices_round_1_day_-1.csv", f"{DATA_DIR}/trades_round_1_day_-1.csv")
p.visualize_orderbook(product="INTARIAN_PEPPER_ROOT", ymin=11000, ymax=11500, renderer="browser")

## Day 0

In [ ]:
p = Plotter(f"{DATA_DIR}/prices_round_1_day_0.csv", f"{DATA_DIR}/trades_round_1_day_0.csv")
p.visualize_orderbook(product="INTARIAN_PEPPER_ROOT", ymin=10000, ymax=10500, renderer="browser")

## All Days Merged

In [4]:
p = Plotter(
    [f"{DATA_DIR}/prices_round_1_day_{d}.csv" for d in days],
    [f"{DATA_DIR}/trades_round_1_day_{d}.csv" for d in days],
)
p.visualize_orderbook(product="INTARIAN_PEPPER_ROOT", ymin=10000, ymax=13000, renderer="browser")

In [8]:
p.visualize_orderbook(product="ASH_COATED_OSMIUM", renderer="browser", ymin = 9975, ymax = 10030)

## Backtest Log Visualization

In [ ]:
from plotter import LogVisualizer

viz = LogVisualizer("../backtests/2026-04-14_21-42-49.log")
viz.summary()
viz.visualize(renderer="browser")